# Célula de configurações

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, trim, lower, regexp_replace

os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

iceberg_version = "1.4.3"
aws_version = "3.3.4"

packages = [
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{iceberg_version}",
    f"org.apache.iceberg:iceberg-aws-bundle:{iceberg_version}",
    f"org.apache.hadoop:hadoop-aws:{aws_version}",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

spark = SparkSession.builder \
    .appName("Ingestao-Lakehouse-CGE") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


print("Spark configurado e conectado ao Iceberg/MinIO!")

# Criação de catalogo e banco de dados no Iceberg

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.silver")
print("Namespace 'iceberg.silver' verificado/criado.")

Namespace 'iceberg.silver' verificado/criado.


# Leitura do arquivo de ingestão

In [ ]:
file_path = "/workspaces/laboratorio-lakehouse/data/dados_raw.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .csv(file_path)

print(f"Total de registros lidos: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

# Transformação e limpeza dos dados 

In [ ]:
def clean_column_names(df):
    for col_name in df.columns:
        clean_name = col_name.strip().replace(" ", "_").lower()
        df = df.withColumnRenamed(col_name, clean_name)
    return df

df_clean = clean_column_names(df_raw)

df_silver = df_clean.withColumn("data_ingestao", current_timestamp())

print("Schema após o refinamento:")
df_silver.printSchema()

# Migração dos dados 

In [ ]:
table_name = "iceberg.silver.dados_auditoria"

df_silver.writeTo(table_name) \
    .tableProperty("format-version", "2") \
    .using("iceberg") \
    .createOrReplace()

print(f"Tabela {table_name} gravada com sucesso no MinIO/Iceberg!")

# Vizualização dos dados escritos

In [ ]:
spark.sql(f"SELECT * FROM {table_name} LIMIT 5").show()

spark.sql(f"SELECT * FROM {table_name}.snapshots").show(truncate=False)